# Representasi Permutasi Latin Square

Program ini:
1. Generate semua Latin square order $n$
2. Ekstrak representasi permutasi untuk setiap elemen (sesuai Proposisi 2.6.1)
3. Export hasil ke file LaTeX

**Proposisi 2.6.1**: Setiap Latin square $A \in L_S(n)$ dengan simbol $\{a_1, a_2, \ldots, a_n\}$ dapat dituliskan sebagai:
$$A = \bigoplus_{i=1}^{n} a_i P_{\tau_i}$$
dimana $P_{\tau_i}$ adalah matriks permutasi max-plus yang bersesuaian dengan permutasi $\tau_i$.

In [42]:
"""
Latin Square Library - Reusable untuk berbagai order
"""
import numpy as np
from typing import List, Dict, Tuple
import os

class LatinSquare:
    """Class untuk merepresentasikan Latin Square dan operasinya"""
    
    def __init__(self, matrix: np.ndarray, symbols: List[int] = None):
        self.matrix = np.array(matrix, dtype=int)
        self.n = len(matrix)
        self.symbols = symbols if symbols else sorted(np.unique(matrix).tolist())
    
    def __repr__(self):
        return f"LatinSquare(n={self.n})\n{self.matrix}"
    
    def __eq__(self, other):
        return np.array_equal(self.matrix, other.matrix)
    
    def __hash__(self):
        return hash(self.matrix.tobytes())

class LatinSquareGenerator:
    """Generator Latin Square menggunakan backtracking dengan constraint propagation"""
    
    def __init__(self, n: int, symbols: List[int] = None):
        self.n = n
        self.symbols = symbols if symbols else list(range(1, n + 1))
        self.all_squares = []
    
    def generate_all(self) -> List[LatinSquare]:
        """Generate semua Latin square order n dengan constraint propagation"""
        self.all_squares = []
        
        # Initialize empty grid dengan possible values
        grid = [[0] * self.n for _ in range(self.n)]
        possible = [[set(self.symbols) for _ in range(self.n)] for _ in range(self.n)]
        
        self._backtrack(grid, possible, 0, 0)
        
        return self.all_squares
    
    def _backtrack(self, grid, possible, row, col):
        """Backtracking dengan constraint propagation"""
        if row == self.n:
            # Found valid Latin square
            self.all_squares.append(
                LatinSquare(np.array(grid), self.symbols.copy())
            )
            return
        
        next_row, next_col = (row, col + 1) if col + 1 < self.n else (row + 1, 0)
        
        for val in list(possible[row][col]):
            # Place value
            grid[row][col] = val
            
            # Save state for backtracking
            removed = []
            
            # Propagate constraint: remove val from same row and column
            valid = True
            for k in range(self.n):
                if k != col and val in possible[row][k]:
                    possible[row][k].remove(val)
                    removed.append((row, k, val))
                    if not possible[row][k] and grid[row][k] == 0:
                        valid = False
                if k != row and val in possible[k][col]:
                    possible[k][col].remove(val)
                    removed.append((k, col, val))
                    if not possible[k][col] and grid[k][col] == 0:
                        valid = False
            
            if valid:
                self._backtrack(grid, possible, next_row, next_col)
            
            # Restore state
            grid[row][col] = 0
            for r, c, v in removed:
                possible[r][c].add(v)
    
    def count(self) -> int:
        """Return jumlah Latin square yang sudah di-generate"""
        return len(self.all_squares)

print("✓ Library LatinSquare berhasil dimuat!")
print("  - LatinSquare: Class untuk representasi Latin square")
print("  - LatinSquareGenerator: Class untuk generate semua Latin square")

✓ Library LatinSquare berhasil dimuat!
  - LatinSquare: Class untuk representasi Latin square
  - LatinSquareGenerator: Class untuk generate semua Latin square


In [43]:
class PermutationExtractor:
    """
    Ekstrak representasi permutasi dari Latin square (Proposisi 2.6.1)
    
    Untuk setiap simbol a_i, definisikan permutasi τ_i dimana:
        τ_i(r) = c  ⟺  [A]_rc = a_i
    
    Artinya: τ_i memetakan baris r ke kolom c dimana elemen a_i berada
    """
    
    @staticmethod
    def extract_permutations(ls: LatinSquare) -> Dict[int, Tuple[int, ...]]:
        """
        Ekstrak permutasi untuk setiap simbol dalam Latin square
        
        Returns:
            Dict mapping simbol -> permutasi (dalam bentuk tuple)
            Permutasi τ[r] = c berarti simbol berada di kolom c untuk baris r
        """
        n = ls.n
        permutations_dict = {}
        
        for symbol in ls.symbols:
            perm = [0] * n  # τ(0), τ(1), ..., τ(n-1)
            for r in range(n):
                for c in range(n):
                    if ls.matrix[r, c] == symbol:
                        perm[r] = c
                        break
            permutations_dict[symbol] = tuple(perm)
        
        return permutations_dict
    
    @staticmethod
    def extract_permutation_matrices(ls: LatinSquare) -> Dict[int, np.ndarray]:
        """
        Ekstrak matriks permutasi untuk setiap simbol
        
        Returns:
            Dict mapping simbol -> matriks permutasi P_τ
            P_τ[i,j] = 1 jika τ(i) = j, 0 otherwise
        """
        n = ls.n
        perm_matrices = {}
        
        for symbol in ls.symbols:
            P = np.zeros((n, n), dtype=int)
            for r in range(n):
                for c in range(n):
                    if ls.matrix[r, c] == symbol:
                        P[r, c] = 1
            perm_matrices[symbol] = P
        
        return perm_matrices
    
    @staticmethod
    def permutation_to_cycle_notation(perm: Tuple[int, ...], one_indexed: bool = True) -> str:
        """
        Convert permutasi ke notasi siklus
        
        Args:
            perm: Tuple permutasi dimana perm[i] = j berarti i -> j
            one_indexed: Jika True, gunakan 1,2,3,... (untuk LaTeX)
        
        Returns:
            String notasi siklus, misal "(1 2 3)" atau "(1 3)(2)"
        """
        n = len(perm)
        offset = 1 if one_indexed else 0
        visited = [False] * n
        cycles = []
        
        for start in range(n):
            if visited[start]:
                continue
            
            cycle = []
            current = start
            while not visited[current]:
                visited[current] = True
                cycle.append(current + offset)
                current = perm[current]
            
            if len(cycle) > 1:
                cycles.append(cycle)
            elif len(cycle) == 1:
                # Fixed point - bisa ditampilkan atau tidak
                cycles.append(cycle)
        
        if not cycles:
            return "()"
        
        return "".join(f"({' '.join(map(str, c))})" for c in cycles)

print("✓ PermutationExtractor berhasil dimuat!")
print("  - extract_permutations: Ekstrak permutasi dari Latin square")
print("  - extract_permutation_matrices: Ekstrak matriks permutasi")
print("  - permutation_to_cycle_notation: Konversi ke notasi siklus")

✓ PermutationExtractor berhasil dimuat!
  - extract_permutations: Ekstrak permutasi dari Latin square
  - extract_permutation_matrices: Ekstrak matriks permutasi
  - permutation_to_cycle_notation: Konversi ke notasi siklus


In [44]:
class LaTeXExporter:
    """Export Latin square dan representasi permutasinya ke LaTeX (format ringkas untuk twocolumn)"""
    
    def __init__(self, output_dir: str = "."):
        self.output_dir = output_dir
    
    def matrix_to_latex(self, matrix: np.ndarray, env: str = "pmatrix") -> str:
        """Convert numpy matrix ke LaTeX matrix environment"""
        n = matrix.shape[0]
        rows = []
        for i in range(n):
            row_str = " & ".join(str(int(matrix[i, j])) for j in range(n))
            rows.append(row_str)
        
        content = " \\\\ ".join(rows)
        return f"\\begin{{{env}}} {content} \\end{{{env}}}"
    
    def cycle_notation_to_latex(self, perm: Tuple[int, ...], one_indexed: bool = True) -> str:
        """Convert permutasi ke notasi siklus LaTeX"""
        return PermutationExtractor.permutation_to_cycle_notation(perm, one_indexed)
    
    def export_latin_square(self, ls: LatinSquare, index: int) -> str:
        """Export satu Latin square ke LaTeX (format kompak dengan matrix untuk permutasi)"""
        perms = PermutationExtractor.extract_permutations(ls)
        
        # Buat matrix untuk permutasi: tau_1 \\ tau_2 \\ tau_3
        perm_rows = " \\\\ ".join(
            f"\\tau_{{{s}}} = {self.cycle_notation_to_latex(perms[s])}" 
            for s in ls.symbols
        )
        
        # Format: L_i = matrix, begin{matrix} permutasi \end{matrix}
        return f"$L_{{{index}}} = {self.matrix_to_latex(ls.matrix)}, \\quad \\begin{{matrix}} {perm_rows} \\end{{matrix}}$\n"
    
    def export_all(self, squares: List[LatinSquare], 
                   filename: str,
                   title: str = None) -> str:
        """Export semua Latin square ke file .tex (format ringkas)"""
        lines = []
        
        n = squares[0].n if squares else 0
        symbols_str = ", ".join(map(str, squares[0].symbols)) if squares else ""
        
        # Header comment
        lines.append(f"% File: {filename}.tex")
        lines.append(f"% Latin Squares order {n}, symbols {{{symbols_str}}}, total: {len(squares)}")
        lines.append("")
        
        if title:
            lines.append(f"\\subsection*{{{title}}}")
        else:
            lines.append(f"\\subsection*{{Latin Square Order {n}}}")
        lines.append("")
        
        lines.append(f"Terdapat {len(squares)} Latin square berbeda dengan order {n} dan simbol $\\{{{symbols_str}\\}}$:")
        lines.append("")
        
        # Export each Latin square
        for i, ls in enumerate(squares, 1):
            lines.append(self.export_latin_square(ls, i))
        
        # Write to file
        filepath = os.path.join(self.output_dir, f"{filename}.tex")
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        
        return filepath

print("✓ LaTeXExporter berhasil dimuat (format kompak untuk twocolumn)!")

✓ LaTeXExporter berhasil dimuat (format kompak untuk twocolumn)!


## Contoh: Generate Latin Square Order 3

Menggunakan simbol $\{1, 2, 3\}$

In [45]:
# ============================================================
# KONFIGURASI - Ubah ini untuk order yang berbeda
# ============================================================
ORDER = 4
SYMBOLS = list(range(1, ORDER + 1)) 

# ============================================================
# Generate semua Latin square
# ============================================================
print(f"{'='*60}")
print(f"GENERATING LATIN SQUARES ORDER {ORDER}")
print(f"Simbol: {{{', '.join(map(str, SYMBOLS))}}}")
print(f"{'='*60}")

generator = LatinSquareGenerator(ORDER, SYMBOLS)
all_squares = generator.generate_all()

print(f"\n✓ Ditemukan {len(all_squares)} Latin square")
print(f"\nContoh Latin square pertama:")
print(all_squares[0].matrix)

GENERATING LATIN SQUARES ORDER 4
Simbol: {1, 2, 3, 4}

✓ Ditemukan 576 Latin square

Contoh Latin square pertama:
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]


In [46]:
# ============================================================
# Tampilkan semua Latin square dengan representasi permutasinya
# ============================================================
print(f"\n{'='*60}")
print("SEMUA LATIN SQUARE DAN REPRESENTASI PERMUTASI")
print(f"{'='*60}")

for i, ls in enumerate(all_squares, 1):
    print(f"\n--- Latin Square #{i} ---")
    print(ls.matrix)
    
    perms = PermutationExtractor.extract_permutations(ls)
    print("\nRepresentasi Permutasi:")
    for symbol in SYMBOLS:
        perm = perms[symbol]
        cycle = PermutationExtractor.permutation_to_cycle_notation(perm)
        # Tampilkan dalam format: τ_symbol: (baris) -> (kolom)
        mapping = ", ".join(f"{r+1}→{perm[r]+1}" for r in range(ORDER))
        print(f"  τ_{symbol}: {mapping}  =  {cycle}")
    
    print()


SEMUA LATIN SQUARE DAN REPRESENTASI PERMUTASI

--- Latin Square #1 ---
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]

Representasi Permutasi:
  τ_1: 1→1, 2→2, 3→3, 4→4  =  (1)(2)(3)(4)
  τ_2: 1→2, 2→1, 3→4, 4→3  =  (1 2)(3 4)
  τ_3: 1→3, 2→4, 3→1, 4→2  =  (1 3)(2 4)
  τ_4: 1→4, 2→3, 3→2, 4→1  =  (1 4)(2 3)


--- Latin Square #2 ---
[[1 2 3 4]
 [2 1 4 3]
 [3 4 2 1]
 [4 3 1 2]]

Representasi Permutasi:
  τ_1: 1→1, 2→2, 3→4, 4→3  =  (1)(2)(3 4)
  τ_2: 1→2, 2→1, 3→3, 4→4  =  (1 2)(3)(4)
  τ_3: 1→3, 2→4, 3→1, 4→2  =  (1 3)(2 4)
  τ_4: 1→4, 2→3, 3→2, 4→1  =  (1 4)(2 3)


--- Latin Square #3 ---
[[1 2 3 4]
 [2 1 4 3]
 [4 3 1 2]
 [3 4 2 1]]

Representasi Permutasi:
  τ_1: 1→1, 2→2, 3→3, 4→4  =  (1)(2)(3)(4)
  τ_2: 1→2, 2→1, 3→4, 4→3  =  (1 2)(3 4)
  τ_3: 1→3, 2→4, 3→2, 4→1  =  (1 3 2 4)
  τ_4: 1→4, 2→3, 3→1, 4→2  =  (1 4 2 3)


--- Latin Square #4 ---
[[1 2 3 4]
 [2 1 4 3]
 [4 3 2 1]
 [3 4 1 2]]

Representasi Permutasi:
  τ_1: 1→1, 2→2, 3→4, 4→3  =  (1)(2)(3 4)
  τ_2: 1→2, 2→1, 3→3, 4→4  =  (1

## Export ke LaTeX

Export hasil ke file `.tex` yang bisa di-import dengan `\input{}`

In [47]:
# ============================================================
# Export ke LaTeX
# ============================================================
exporter = LaTeXExporter(output_dir=".")

output_file = f"latin_square_order_{ORDER}"
filepath = exporter.export_all(
    all_squares,
    filename=output_file,
    title=f"Latin Square Order {ORDER}"
)

print(f"✓ File LaTeX berhasil dibuat: {filepath}")
print(f"  Total: {len(all_squares)} Latin square")
print(f"\nUntuk import di main.tex, gunakan:")
print(f"  \\input{{{output_file}}}")

✓ File LaTeX berhasil dibuat: .\latin_square_order_4.tex
  Total: 576 Latin square

Untuk import di main.tex, gunakan:
  \input{latin_square_order_4}


In [48]:
# ============================================================
# Preview isi file LaTeX yang dihasilkan
# ============================================================
print("="*60)
print("PREVIEW FILE LATEX")
print("="*60)

with open(filepath, 'r', encoding='utf-8') as f:
    content = f.read()
    # Tampilkan 80 baris pertama
    lines = content.split('\n')
    preview_lines = lines[:80]
    print('\n'.join(preview_lines))
    if len(lines) > 80:
        print(f"\n... ({len(lines) - 80} baris lagi)")

PREVIEW FILE LATEX
% File: latin_square_order_4.tex
% Latin Squares order 4, symbols {1, 2, 3, 4}, total: 576

\subsection*{Latin Square Order 4}

Terdapat 576 Latin square berbeda dengan order 4 dan simbol $\{1, 2, 3, 4\}$:

$L_{1} = \begin{pmatrix} 1 & 2 & 3 & 4 \\ 2 & 1 & 4 & 3 \\ 3 & 4 & 1 & 2 \\ 4 & 3 & 2 & 1 \end{pmatrix}, \quad \begin{matrix} \tau_{1} = (1)(2)(3)(4) \\ \tau_{2} = (1 2)(3 4) \\ \tau_{3} = (1 3)(2 4) \\ \tau_{4} = (1 4)(2 3) \end{matrix}$

$L_{2} = \begin{pmatrix} 1 & 2 & 3 & 4 \\ 2 & 1 & 4 & 3 \\ 3 & 4 & 2 & 1 \\ 4 & 3 & 1 & 2 \end{pmatrix}, \quad \begin{matrix} \tau_{1} = (1)(2)(3 4) \\ \tau_{2} = (1 2)(3)(4) \\ \tau_{3} = (1 3)(2 4) \\ \tau_{4} = (1 4)(2 3) \end{matrix}$

$L_{3} = \begin{pmatrix} 1 & 2 & 3 & 4 \\ 2 & 1 & 4 & 3 \\ 4 & 3 & 1 & 2 \\ 3 & 4 & 2 & 1 \end{pmatrix}, \quad \begin{matrix} \tau_{1} = (1)(2)(3)(4) \\ \tau_{2} = (1 2)(3 4) \\ \tau_{3} = (1 3 2 4) \\ \tau_{4} = (1 4 2 3) \end{matrix}$

$L_{4} = \begin{pmatrix} 1 & 2 & 3 & 4 \\ 2 & 1 & 4 & 3 

## Cara Menggunakan untuk Order Lain

Untuk generate Latin square dengan order berbeda, cukup ubah nilai `ORDER` di cell konfigurasi. Contoh:

```python
# Untuk order 4
ORDER = 4
SYMBOLS = list(range(1, ORDER + 1))  # [1, 2, 3, 4]

# Untuk order 5 (membutuhkan waktu lebih lama)
ORDER = 5
SYMBOLS = list(range(1, ORDER + 1))  # [1, 2, 3, 4, 5]
```

**Jumlah Latin Square per Order:**

| Order | Jumlah |
|-------|--------|
| 1 | 1 |
| 2 | 2 |
| 3 | 12 |
| 4 | 576 |
| 5 | 161,280 |

## Pasangan Latin Square Komutatif (Max-Plus)

Mencari semua pasangan $(A, B)$ dengan $A \neq B$ yang memenuhi: $A \otimes B = B \otimes A$

In [49]:
# ============================================================
# Fungsi Max-Plus dan Pencarian Pasangan Komutatif
# ============================================================

def maxplus_mult(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Perkalian max-plus: (A ⊗ B)_ij = max_k(A_ik + B_kj)"""
    n = A.shape[0]
    C = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            C[i, j] = max(A[i, k] + B[k, j] for k in range(n))
    return C

def matrices_equal(A: np.ndarray, B: np.ndarray) -> bool:
    """Cek apakah dua matriks sama"""
    return np.allclose(A, B)

def find_commutative_pairs(squares: List[LatinSquare]) -> List[dict]:
    """
    Cari semua pasangan Latin square (A, B) dengan A ≠ B yang komutatif: A ⊗ B = B ⊗ A
    """
    commutative_pairs = []
    n_squares = len(squares)
    
    for i in range(n_squares):
        for j in range(i + 1, n_squares):  # j > i agar tidak duplikat
            A = squares[i].matrix
            B = squares[j].matrix
            
            AB = maxplus_mult(A, B)
            BA = maxplus_mult(B, A)
            
            if matrices_equal(AB, BA):
                commutative_pairs.append({
                    'A': squares[i],
                    'B': squares[j],
                    'A_index': i + 1,
                    'B_index': j + 1,
                    'product': AB
                })
    
    return commutative_pairs

# Cari pasangan komutatif
print(f"{'='*60}")
print(f"MENCARI PASANGAN KOMUTATIF (A ⊗ B = B ⊗ A)")
print(f"{'='*60}")

commutative_pairs = find_commutative_pairs(all_squares)

print(f"\n✓ Ditemukan {len(commutative_pairs)} pasangan komutatif")
print(f"  dari total {len(all_squares) * (len(all_squares) - 1) // 2} pasangan yang dicek")

MENCARI PASANGAN KOMUTATIF (A ⊗ B = B ⊗ A)

✓ Ditemukan 4752 pasangan komutatif
  dari total 165600 pasangan yang dicek


In [50]:
# ============================================================
# Export Pasangan Komutatif ke LaTeX (format multicol)
# ============================================================

def export_commutative_pairs(pairs: List[dict], 
                              filename: str,
                              order: int,
                              output_dir: str = ".",
                              num_cols: int = 2) -> str:
    """Export pasangan komutatif ke file .tex dengan format multicol"""
    lines = []
    
    # Matrix to latex helper (compact)
    def mat_to_tex(m):
        rows = " \\\\ ".join(" & ".join(str(int(m[i,j])) for j in range(m.shape[1])) for i in range(m.shape[0]))
        return f"\\begin{{pmatrix}} {rows} \\end{{pmatrix}}"
    
    # Permutation to matrix format (vertical stacking, scriptsize)
    def perms_to_matrix_small(perms: dict, symbols: list) -> str:
        rows = " \\\\ ".join(
            f"\\tau_{{{s}}}={PermutationExtractor.permutation_to_cycle_notation(perms[s])}"
            for s in symbols
        )
        return f"\\begin{{matrix}} {rows} \\end{{matrix}}"
    
    # Header
    lines.append(f"% File: {filename}.tex")
    lines.append(f"% Commutative pairs of Latin squares order {order}")
    lines.append(f"% Total: {len(pairs)} pairs")
    lines.append("")
    
    lines.append(f"\\subsection*{{Pasangan Latin Square Komutatif Order {order}}}")
    lines.append("")
    lines.append(f"Terdapat {len(pairs)} pasangan $(A, B)$ dengan $A \\neq B$ yang memenuhi $A \\otimes B = B \\otimes A$:")
    lines.append("")
    lines.append(f"\\begin{{multicols}}{{{num_cols}}}")
    lines.append("\\small")
    lines.append("")
    
    for idx, pair in enumerate(pairs, 1):
        A = pair['A']
        B = pair['B']
        product = pair['product']
        
        # Ekstrak permutasi
        perms_A = PermutationExtractor.extract_permutations(A)
        perms_B = PermutationExtractor.extract_permutations(B)
        
        # Format permutasi (scriptsize matrix)
        perm_A_matrix = perms_to_matrix_small(perms_A, A.symbols)
        perm_B_matrix = perms_to_matrix_small(perms_B, B.symbols)
        
        lines.append(f"\\noindent\\textbf{{{idx}.}} $L_{{{pair['A_index']}}} \\circ L_{{{pair['B_index']}}}$")
        lines.append("")
        # Matriks dan permutasi di samping (satu baris)
        lines.append(f"$A = {mat_to_tex(A.matrix)} \\quad {perm_A_matrix}$")
        lines.append("")
        lines.append(f"$B = {mat_to_tex(B.matrix)} \\quad {perm_B_matrix}$")
        lines.append("")
        lines.append(f"$A \\otimes B = {mat_to_tex(product)}$")
        lines.append("")
        lines.append("\\vspace{0.3em}")
        lines.append("\\hrule")
        lines.append("\\vspace{0.3em}")
        lines.append("")
    
    lines.append("\\end{multicols}")
    
    # Write to file
    filepath = os.path.join(output_dir, f"{filename}.tex")
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write("\n".join(lines))
    
    return filepath

# Export (2 kolom untuk order 3, bisa ubah num_cols sesuai kebutuhan)
output_file_comm = f"latin_square_commutative_pairs_order_{ORDER}"
filepath_comm = export_commutative_pairs(
    commutative_pairs,
    filename=output_file_comm,
    order=ORDER,
    num_cols=3
)

print(f"✓ File LaTeX berhasil dibuat: {filepath_comm}")
print(f"  Total: {len(commutative_pairs)} pasangan komutatif")
print(f"\nUntuk import di main.tex, gunakan:")
print(f"  \\input{{{output_file_comm}}}")

✓ File LaTeX berhasil dibuat: .\latin_square_commutative_pairs_order_4.tex
  Total: 4752 pasangan komutatif

Untuk import di main.tex, gunakan:
  \input{latin_square_commutative_pairs_order_4}
